In [ ]:
!pip install langchain
!pip install langchain-community
!pip install langchain-groq
!pip install langchain-google-genai
!pip install langchain-openai
!pip install python-dotenv

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [ ]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.3-70b-versatile", api_key=GROQ_API_KEY)

## Creation of Tools

We use tool from langchain.tools and it act as a decorater, if we mention it above any funtion like "@tool" then it become a tool in langchain

The Docstring plays a very important role in tool creation because of docstring the llm is able to identify the functionality of this tool.

In [ ]:
from langchain.tools import tool

@tool
def get_weather(city:str) -> str:
  """Get weather for a city"""  # <- This is Docstring
  return f"The weather in the {city} is Sunny"

Now we have to bind this tool with our model, one way is the same we did earlier with the help of "create_agent" from "langchain.agents". The other way is to use bind_tools() function, and it also takes input a list of tools. The difference is bind_tools() only tells the llm or make the llm know what tools are present it doesnt call it. While on the other side "create_agent()" is high level API it creates a StateGraph with the help of LangGraph which auto manages calling of the tools.

Note here if we using create_agent() we dont need to specify "@tool" in above functions because it is build on Langgraph and it automatically creates a ToolNode wiht a StateGraph in Langgraph, but for bind_tools() it is mandatory to decorate the functions with "@tool" because it expects a Tool object

In [ ]:
model_with_tools = model.bind_tools([get_weather])

In [ ]:
response = model_with_tools.invoke("What is weather in Delhi")

print(response)

for tool_call in response.tool_calls:
  print("Tools : ", {tool_call["name"]})

content='' additional_kwargs={'tool_calls': [{'id': '59pjs0cdp', 'function': {'arguments': '{"city":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 215, 'total_tokens': 230, 'completion_time': 0.045769451, 'completion_tokens_details': None, 'prompt_time': 0.029113719, 'prompt_tokens_details': None, 'queue_time': 0.070673067, 'total_time': 0.07488317}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_43d97c5965', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019f1d88-4ead-71d1-84df-6d29312d27f0-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': '59pjs0cdp', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 215, 'output_tokens': 15, 'total_tokens': 230}
Tools :  {'get_weather'}


Note: we are getting nothing in content but some metadata of tool calls, we can use this in tool execution loop

Notice : No tool has been called yet, the llm is telling to call tool in the metadata this is what StateGraph do in langgraph, it is responsible for calling the respective tool requested by the llm

And below here this Tool Execution Loop is just a manual function of that, we are calling tool and appending result to the messages and then again calling the llm for the output

## Tool Execution loop

Basically we are appending the response we got from tool call to our message and passing it again to the model_with_tools

In [ ]:
question = [{
    "role": "user",
    "content": "What is weather in Delhi"
}]
response = model_with_tools.invoke(question)
question.append(response)

# Now at this stage our question also have the response generated by the model which is nothing but some metadata

for tool_call in response.tool_calls:
  tool_result = get_weather.invoke(tool_call)
  question.append(tool_result)

final_response = model_with_tools.invoke(question)
print(final_response.content)

The weather in Delhi is sunny.


In [ ]:
question


[{'role': 'user', 'content': 'What is weather in Delhi'},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm1namjdts', 'function': {'arguments': '{"city":"Delhi"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 215, 'total_tokens': 230, 'completion_time': 0.045894966, 'completion_tokens_details': None, 'prompt_time': 0.014432249, 'prompt_tokens_details': None, 'queue_time': 0.145800573, 'total_time': 0.060327215}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_ce7bc1685b', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f1d92-f2f1-7970-801c-2aacf1e0cd45-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Delhi'}, 'id': 'm1namjdts', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 215, 'output_tokens': 15, 'total_tokens': 230}),
 ToolMessage(content='The weather in the 